# 12. TWOSIDES DDI (via PyTDC) EDA (T0-3)

**데이터 출처**: Therapeutics Data Commons (TDC) — `tdc.multi_pred.DDI`  
**파이프라인 역할**: Stage 2 DDI 검색 베이스라인 / 국제 데이터 매핑 가능성 평가

## EDA 체크리스트
- [ ] 약물 수, DDI pair 수
- [ ] 한국 낱알식별 성분명과 매핑되는 약물 수 → DDI 재현율 상한선
- [ ] pair 수 분포 (long-tail; FAISS 크기 추정)
- [ ] 대칭성 확인 (A→B vs B→A)
- [ ] 산출물: `data/interim/twosides_clean.parquet` + `twosides_summary.json`

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("../../").resolve()
INTERIM = ROOT / "data" / "interim"

In [ ]:
# TWOSIDES 데이터 로드
from tdc.multi_pred import DDI

data = DDI(name="TWOSIDES", path=str(ROOT / "data" / "raw" / "tdc"))
split = data.get_split()

# train/valid/test 합산
df = pd.concat([split["train"], split["valid"], split["test"]], ignore_index=True)
print(f"shape: {df.shape}")
print(df.dtypes)
df.head(3)

In [ ]:
# 컬럼명 확인 후 drug1/drug2 식별
print(df.columns.tolist())

In [ ]:
# 약물 수 / pair 수
drug_cols = [c for c in df.columns if "drug" in c.lower() or "smiles" in c.lower()]
id_col1, id_col2 = drug_cols[0], drug_cols[1] if len(drug_cols) >= 2 else (drug_cols[0], drug_cols[0])

all_drugs = pd.concat([df[id_col1], df[id_col2]]).unique()
print(f"유일 약물 수: {len(all_drugs):,}")
print(f"DDI pair 수: {len(df):,}")

In [ ]:
# 약물별 pair 수 분포
pair_counts = pd.concat([
    df[id_col1].value_counts(),
    df[id_col2].value_counts()
]).groupby(level=0).sum()

print(pair_counts.describe())
pair_counts.hist(bins=50, figsize=(10, 4))
plt.title("약물별 DDI pair 수 분포")
plt.xlabel("pair 수")
plt.show()

In [ ]:
# 대칭성 확인
forward = set(zip(df[id_col1], df[id_col2]))
backward = set(zip(df[id_col2], df[id_col1]))
symmetric = forward & backward
print(f"대칭 pair: {len(symmetric):,} / 전체: {len(forward):,} ({len(symmetric)/len(forward):.1%})")

In [ ]:
# 한국 성분명과 매핑 시도
nadal_path = INTERIM / "nadal_ident_clean.parquet"
if nadal_path.exists():
    df_nadal = pd.read_parquet(nadal_path)
    # 성분명 컬럼 후보 탐색
    ingr_cols = [c for c in df_nadal.columns if "INGR" in c.upper() or "INGDT" in c.upper() or "COMP" in c.upper()]
    print("낱알식별 성분명 관련 컬럼:", ingr_cols)
    # 상세 매핑은 01_korean_id_ontology.ipynb에서 체계적으로 수행
else:
    print("낱알식별 parquet 없음 — 10_nadal_ident 먼저 실행")

In [ ]:
# 산출물 저장
df.to_parquet(INTERIM / "twosides_clean.parquet", index=False)

summary = {
    "total_pairs": len(df),
    "unique_drugs": int(len(all_drugs)),
    "columns": df.columns.tolist(),
    "symmetric_pair_rate": float(len(symmetric) / len(forward)),
}
with open(INTERIM / "twosides_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("저장 완료")
print(json.dumps(summary, indent=2))